# SYS-37 — LLM Dataset Hazırlığı
**Branch:** `SYS-37-LLM-Dataset`  
**Amaç:** Ham donanım CSV dosyalarını temizle, normalize et ve ML modeli için feature engineering uygula.

### İşlenecek verisetleri ve dosyalar
#### Veri Seti ve Kullanım Yeri 
- gpus-fps-on-games baraazaid → FPS etiketi Ana model 
- fps-benchmark ulrikthygepedersen → Çözünürlük/ayar Ana model 
- gpu-benchmarks alanjo → G3D Mark skoru Ana model 
- featurecpu-benchmarks → alanjo CPU Mark skoru Darboğaz hesabı 
- general-computer-hardware → dilshaansandhu Specs + fiyat PSU, RAM, disk araçları

#### Dosyalar
- `CPUData.csv` → CPU benchmark tahmini için
- `GPUData.csv` → GPU performans tahmini için  
- `RAMData.csv` → RAM latency/bandwidth hesabı için
- `PSUData.csv` → PSU watt hesaplayıcısı için
- `HDDData.csv`, `SSDData.csv` → Depolama karşılaştırması için

In [ ]:
import os

print("--- Kaggle Input Klasöründeki Tüm Dosyalar ---")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

---
## 1. Veri Setlerinin yüklenmesi

In [ ]:
import pandas as pd

# 1. Ana Model - FPS Etiketleri (JSON formatında!)
df_gpus_fps = pd.read_json('/kaggle/input/datasets/baraazaid/gpus-fps-on-games/gpus.json')

# 2. Ana Model - Çözünürlük ve Ayar Verisi
df_fps_bench = pd.read_csv('/kaggle/input/datasets/ulrikthygepedersen/fps-benchmark/fps_benchmark.csv')

# 3. Ana Model Feature - GPU Benchmark (v7 versiyonunu alıyoruz)
df_gpu_bench = pd.read_csv('/kaggle/input/datasets/alanjo/gpu-benchmarks/GPU_benchmarks_v7.csv')

# 4. Darboğaz Hesabı - CPU Benchmark (v4 versiyonunu alıyoruz)
df_cpu_bench = pd.read_csv('/kaggle/input/datasets/alanjo/cpu-benchmarks/CPU_benchmark_v4.csv')

# 5. Araçlar İçin Donanım Verileri (Şimdilik örnek olarak PSU ve RAM'i yüklüyoruz)
df_psu = pd.read_csv('/kaggle/input/datasets/dilshaansandhu/general-computer-hardware-dataset/PSUData.csv')
df_ram = pd.read_csv('/kaggle/input/datasets/dilshaansandhu/general-computer-hardware-dataset/RAMData.csv')

print("Bütün veri setleri başarıyla yüklendi!")

---
## 2. Toplam Veri türü

In [ ]:
print("--- GPU FPS JSON Verisi ---")
print(df_gpus_fps.head(2))
print(df_gpus_fps.columns)

print("\n--- Çözünürlük ve Ayar Verisi ---")
print(df_fps_bench.head(2))
print(df_fps_bench.columns)

print("\n--- GPU Benchmark Verisi ---")
print(df_gpu_bench.head(2))
print(df_gpu_bench.columns)

---
## 2. Verilerin Temizlenmesi ve tek verisetinde birleştirilmesi

In [ ]:
import pandas as pd
import numpy as np
import ast

# 1. Verileri Yükle
df_fps_bench = pd.read_csv('/kaggle/input/datasets/ulrikthygepedersen/fps-benchmark/fps_benchmark.csv')
df_gpu_bench = pd.read_csv('/kaggle/input/datasets/alanjo/gpu-benchmarks/GPU_benchmarks_v7.csv')
df_cpu_bench = pd.read_csv('/kaggle/input/datasets/alanjo/cpu-benchmarks/CPU_benchmark_v4.csv')

# 2. Temizleme Fonksiyonları
def clean_byte_string(val):
    """ b'metin' şeklindeki verileri normal metne çevirir """
    if isinstance(val, str) and val.startswith("b'") and val.endswith("'"):
        return val[2:-1]
    return str(val)

def create_merge_key(name):
    """ Donanım isimlerini standartlaştırarak (örn: rtx3060ti) eşleşme anahtarı üretir """
    if pd.isna(name): return "unknown"
    name = str(name).lower()
    # Marka ve alt marka isimlerini sil
    remove_words = ['nvidia', 'amd', 'intel', 'geforce', 'radeon', 'core', 'ryzen', ' ', '-']
    for word in remove_words:
        name = name.replace(word, '')
    return name

# 3. FPS Benchmark Verisini Temizle (Ana Tablomuz)
# Byte string'leri temizle
for col in ['CpuName', 'GpuName', 'GameName', 'GameSetting']:
    df_fps_bench[col] = df_fps_bench[col].apply(clean_byte_string)

# Birleştirme anahtarlarını oluştur
df_fps_bench['GPU_Key'] = df_fps_bench['GpuName'].apply(create_merge_key)
df_fps_bench['CPU_Key'] = df_fps_bench['CpuName'].apply(create_merge_key)

# 4. Benchmark Verilerine Anahtar Ekle
df_gpu_bench['GPU_Key'] = df_gpu_bench['gpuName'].apply(create_merge_key)
df_cpu_bench['CPU_Key'] = df_cpu_bench['cpuName'].apply(create_merge_key)

# 5. VERİLERİ BİRLEŞTİRME (MERGE)
# Önce FPS tablosuna GPU benchmark skorlarını ekle
df_main = pd.merge(
    df_fps_bench[['GPU_Key', 'CPU_Key', 'GameName', 'GameResolution', 'GameSetting', 'FPS']], 
    df_gpu_bench[['GPU_Key', 'G3Dmark']], 
    on='GPU_Key', 
    how='left'
)

# Sonra aynı tabloya CPU benchmark skorlarını ekle
df_main = pd.merge(
    df_main, 
    df_cpu_bench[['CPU_Key', 'cpuMark']], 
    on='CPU_Key', 
    how='left'
)

# 6. Sonuçları Kontrol Et ve Dışa Aktar
print("Eksik Benchmark Skoru Olan Satır Sayısı:")
print(df_main.isnull().sum())

# Sadece GPU ve CPU skoru bulunan sağlam verileri filtrele
df_main_clean = df_main.dropna(subset=['G3Dmark', 'cpuMark'])

df_main_clean.to_csv('SYS_37_Main_Model_Dataset.csv', index=False)
print("\nEğitim verisi başarıyla 'SYS_37_Main_Model_Dataset.csv' olarak kaydedildi!")
print(df_main_clean.head(3))

---
## 3. VRAM verisinin eklenmesi

In [ ]:
import ast
import re

# 1. JSON Verisini Tekrar Yükle (Eğer hafızada yoksa)
# df_gpus_fps = pd.read_json('/kaggle/input/datasets/baraazaid/gpus-fps-on-games/gpus.json')

# 2. Karmaşık JSON Hücresinden VRAM (GB) Çeken Fonksiyon
def extract_vram_gb(memory_cell):
    try:
        # Eğer veri string formatında sözlükse, gerçek sözlüğe çevir
        if isinstance(memory_cell, str):
            memory_cell = ast.literal_eval(memory_cell)
        
        # 'Value' anahtarının içindeki metni al (örn: ' 8 GB')
        val_str = memory_cell.get('Value', '')
        
        # Sadece sayıları filtrele
        numbers = re.findall(r'\d+', val_str)
        if numbers:
            return float(numbers[0])
    except:
        return None
    return None

# 3. JSON tablosuna VRAM ve Merge Key ekleme
df_gpus_fps['VRAM_GB'] = df_gpus_fps['Memory'].apply(extract_vram_gb)
df_gpus_fps['GPU_Key'] = df_gpus_fps['Name'].apply(create_merge_key) # Önceki bloktaki fonksiyon

# 4. Ana Tabloya VRAM'i Entegre Etme
# Sadece benzersiz GPU'ları ve VRAM'lerini al (tekrarları önlemek için)
df_vram_unique = df_gpus_fps[['GPU_Key', 'VRAM_GB']].drop_duplicates()

# Final birleştirmesi
df_final_dataset = pd.merge(df_main_clean, df_vram_unique, on='GPU_Key', how='left')

# VRAM'i bilinmeyen (NaN) satırları temizle
df_final_dataset = df_final_dataset.dropna(subset=['VRAM_GB'])

# 5. Artık Eğitime Hazırız!
df_final_dataset.to_csv('SYS_37_Final_Training_Data.csv', index=False)
print("VRAM eklendi! Yeni Sütunlar:")
print(df_final_dataset.columns)
print(df_final_dataset.head(3))